In [1]:
import numpy as np
import random

# --- 1. ENTORNO DEL JUEGO (WHAC-A-MOLE) ---
# Esta clase simula el tablero de juego.

class WhacAMoleEnvironment:
    def __init__(self, grid_size=3, mole_appear_prob=0.3, mole_stay_prob=0.5):
        """
        Inicializa el entorno del juego.
        
        Args:
            grid_size (int): Tamaño del tablero (grid_size x grid_size).
            mole_appear_prob (float): Probabilidad de que aparezca un topo en un agujero vacío.
            mole_stay_prob (float): Probabilidad de que un topo existente se quede en el siguiente turno.
        """
        self.grid_size = grid_size
        self.n_holes = grid_size * grid_size
        self.mole_appear_prob = mole_appear_prob
        self.mole_stay_prob = mole_stay_prob
        
        # El estado es un grid de 0s (vacío) y 1s (topo)
        self.state = np.zeros((grid_size, grid_size), dtype=int)
        
        # Definición de recompensas
        self.reward_hit = 10.0
        self.reward_miss = -5.0
        self.reward_no_action = -1.0 # Penalización pequeña por no hacer nada

    def reset(self):
        """
        Reinicia el juego a su estado inicial.
        Returns:
            numpy.ndarray: El estado inicial del tablero.
        """
        self.state = np.zeros((self.grid_size, self.grid_size), dtype=int)
        return self.state

    def step(self, action):
        """
        Ejecuta una acción en el entorno.
        
        Args:
            action (int): La acción a realizar. Un número del 0 al n_holes-1.
                          Por ejemplo, en un 3x3: 0 es (0,0), 1 es (0,1), ..., 8 es (2,2).
        
        Returns:
            tuple: (next_state, reward, done, info)
                   - next_state (numpy.ndarray): El nuevo estado del tablero.
                   - reward (float): La recompensa de la acción.
                   - done (bool): Si el episodio ha terminado (siempre False en este juego continuo).
                   - info (dict): Información adicional (vacía aquí).
        """
        reward = 0
        
        # Convertir la acción (un número) a coordenadas (fila, columna)
        row = action // self.grid_size
        col = action % self.grid_size
        
        # Comprobar si la acción es válida
        if 0 <= row < self.grid_size and 0 <= col < self.grid_size:
            # Si había un topo, le damos una recompensa positiva y lo quitamos
            if self.state[row, col] == 1:
                reward = self.reward_hit
                self.state[row, col] = 0
            # Si no había topo, recompensa negativa
            else:
                reward = self.reward_miss
        else:
            # Acción inválida
            reward = self.reward_miss

        # --- Actualizar el estado del juego (los topos se mueven) ---
        new_state = np.zeros_like(self.state)
        for r in range(self.grid_size):
            for c in range(self.grid_size):
                # Si hay un topo, decidimos si se queda o se va
                if self.state[r, c] == 1:
                    if random.random() < self.mole_stay_prob:
                        new_state[r, c] = 1
                # Si no hay un topo, decidimos si aparece uno nuevo
                else:
                    if random.random() < self.mole_appear_prob:
                        new_state[r, c] = 1
        
        self.state = new_state
        
        # En este juego, los episodios no tienen un final definido
        done = False 
        info = {}

        return self.state.copy(), reward, done, info

    def render(self):
        """
        Imprime una representación visual del tablero en la consola.
        """
        print("Tablero actual:")
        for row in self.state:
            print(" ".join(['O' if cell == 1 else '.' for cell in row]))
        print("-" * (self.grid_size * 2))

# --- 2. EJEMPLO DE USO DEL ENTORNO (para probarlo manualmente) ---
if __name__ == '__main__':
    print("Iniciando una prueba manual del entorno Whac-A-Mole...")
    
    # Crear el entorno
    env = WhacAMoleEnvironment(grid_size=3)
    
    # Reiniciar el entorno
    state = env.reset()
    env.render()
    
    # Simular 5 turnos de un jugador aleatorio
    for i in range(5):
        # Elegir una acción al azar (golpear un agujero aleatorio)
        action = random.randint(0, env.n_holes - 1)
        print(f"Turno {i+1}: El agente decide golpear la casilla {action} (fila {action//3}, col {action%3})")
        
        # Ejecutar la acción
        next_state, reward, done, info = env.step(action)
        
        print(f"Recompensa obtenida: {reward}")
        env.render()
        
        # Pequeña pausa para que se vea mejor
        # import time
        # time.sleep(1)


Iniciando una prueba manual del entorno Whac-A-Mole...
Tablero actual:
. . .
. . .
. . .
------
Turno 1: El agente decide golpear la casilla 1 (fila 0, col 1)
Recompensa obtenida: -5.0
Tablero actual:
O . .
O . O
O O .
------
Turno 2: El agente decide golpear la casilla 7 (fila 2, col 1)
Recompensa obtenida: 10.0
Tablero actual:
. O .
. . .
O . .
------
Turno 3: El agente decide golpear la casilla 5 (fila 1, col 2)
Recompensa obtenida: -5.0
Tablero actual:
. O .
. . .
. . .
------
Turno 4: El agente decide golpear la casilla 0 (fila 0, col 0)
Recompensa obtenida: -5.0
Tablero actual:
. . O
O . .
O . .
------
Turno 5: El agente decide golpear la casilla 0 (fila 0, col 0)
Recompensa obtenida: -5.0
Tablero actual:
. . O
. . O
. O .
------


In [2]:
# En tu archivo train_agent.py
import numpy as np
import random
from whac_a_mole_env import WhacAMoleEnvironment

# --- Hiperparámetros del Agente ---
EPISODES = 5000
LEARNING_RATE = 0.1
DISCOUNT_FACTOR = 0.95  # Gamma
EPSILON = 1.0  # Tasa de exploración inicial
EPSILON_DECAY = 0.999
MIN_EPSILON = 0.01

# --- Inicialización ---
env = WhacAMoleEnvironment()

# Calcular el número de estados posibles: 2^(n_holes)
n_states = 2 ** env.n_holes 
n_actions = env.n_holes

# Inicializar la tabla Q con ceros
q_table = np.zeros((n_states, n_actions))

# --- Función para convertir el estado (array) a un índice único (int) ---
def state_to_index(state):
    # Convierte el array de numpy a una tupla y luego a un entero
    # Ej: [[1,0],[0,1]] -> (1,0,0,1) -> 1001 (binario) -> 9 (decimal)
    return int(''.join(map(str, state.flatten().tolist())), 2)

# --- Bucle de Entrenamiento ---
for episode in range(EPISODES):
    state = env.reset()
    state_idx = state_to_index(state)
    total_reward = 0
    done = False
    
    while not done:
        # 1. Elegir acción (Epsilon-Greedy)
        if random.uniform(0, 1) < EPSILON:
            action = random.randint(0, n_actions - 1) # Explorar
        else:
            action = np.argmax(q_table[state_idx]) # Explotar
        
        # 2. Ejecutar acción
        next_state, reward, done, _ = env.step(action)
        next_state_idx = state_to_index(next_state)
        
        # 3. Actualizar tabla Q (Fórmula de Q-Learning)
        old_value = q_table[state_idx, action]
        next_max = np.max(q_table[next_state_idx])
        new_value = old_value + LEARNING_RATE * (reward + DISCOUNT_FACTOR * next_max - old_value)
        q_table[state_idx, action] = new_value
        
        # 4. Actualizar estado
        state = next_state
        state_idx = next_state_idx
        total_reward += reward

    # Decaimiento de Epsilon
    if EPSILON > MIN_EPSILON:
        EPSILON *= EPSILON_DECAY

    if (episode + 1) % 100 == 0:
        print(f"Episodio: {episode + 1}, Recompensa Total: {total_reward}, Epsilon: {EPSILON:.2f}")

print("Entrenamiento completado.")

# --- Probar al agente entrenado ---
print("\n--- Probando al agente entrenado (sin exploración) ---")
state = env.reset()
state_idx = state_to_index(state)
env.render()

for _ in range(10):
    action = np.argmax(q_table[state_idx]) # Siempre la mejor acción
    print(f"Agente entrenado golpea la casilla {action}")
    
    next_state, reward, _, _ = env.step(action)
    state_idx = state_to_index(next_state)
    
    print(f"Recompensa: {reward}")
    env.render()

ModuleNotFoundError: No module named 'whac_a_mole_env'